<a href="https://colab.research.google.com/github/GabCAD92/Machine-learning-tasks/blob/main/Gabriel_Owolabi_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Machine Learning
## Supervised Learning: Bayesian Learning

This lesson implements __Naive Bayes__ focusing on a __classification__ task.


### Professor:

<img src="https://www.sorocaba.unesp.br/Home/Graduacao/EngenhariadeControleeAutomacao/alexandre/alex_marta1_small.jpg" width="100" style="float: left; margin-right: 5px;" border="10px" />

  __Prof. Dr. Alexandre da Silva Simões__ <br>
  Control and Automation Engineering Department (DECA) <br>
  Institute of Science and Technology<br>
  São Paulo State University (Unesp) <br>
  Campus Sorocaba <br>
  www.sorocaba.unesp.br/professor/assimoes

<br/>

References:
* Based on: <br/>
  Rodrigues, F. Universidade de São Paulo, São Carlos, Brasil. Ciência de dados: classificador naive Bayes.  https://github.com/franciscorodrigues-usp/ciencia-de-dados/blob/master/Aula4-Classificacao-Naive%20Bayes.ipynb
* Scikit-learn. Naive Bayes. https://scikit-learn.org/stable/modules/naive_bayes.html

____

# Table of Contents


1. Dataset <br>
  1.1. Loading the Iris Dataset <br>
2. Naive Bayes from scratch <br>
	2.1. Likelyhood function <br>
	2.2. Class estimation <br>
	2.3. Accuracy <br>
3. Naive Bayes from scikit-learn <br>
	3.1. Gaussian distribution <br>
	3.2. Bernoulli distribution <br>


____

# 1. Introduction

## 1.1. Loading the Iris Dataset

First of all, let's load the Iris Dataset...

In [1]:
# Import Iris dataset

import random
random.seed(42) # define the seed (important to reproduce the results)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Mount google drive...
from google.colab import drive
drive.mount('/content/drive/')
path = '/content/drive/My Drive/Colab Notebooks/Iris.csv'
f = open(path, 'rb')

# Load data
data = pd.read_csv(f, header=(0))
data = data.dropna(axis='rows') #remove NaN
classes = np.array(pd.unique(data[data.columns[-1]]), dtype=str)  # Store class names

print("Number of lines / rows:", data.shape)
attributes = list(data.columns)
data.head(10)


Mounted at /content/drive/
Number of lines / rows: (150, 5)


,sepal.length,sepal.width,petal.length,petal.width,variety
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa
5,5.4,3.9,1.7,0.4,Setosa
6,4.6,3.4,1.4,0.3,Setosa
7,5.0,3.4,1.5,0.2,Setosa
8,4.4,2.9,1.4,0.2,Setosa
9,4.9,3.1,1.5,0.1,Setosa


In [2]:
# Transform data to Numpy
data = data.to_numpy()
nrow,ncol = data.shape
y = data[:,-1]
X = data[:,0:ncol-1]

The Iris dataset has:

*   3 output classes (setosa, versicolor, virginica)
*   50 examples of each class (150 total examples)
*   4 features (sepal lenght, sepal width, petal lenght, petal width) in cm

<center>
<img src="https://drive.google.com/thumbnail?id=1IpBVgolS9COYB6DdMHLK7efhnVAYznh-" width="250" style="display: block; margin: auto;" border="1px" />
</center>



In [3]:
# Split train and test examples

from sklearn.model_selection import train_test_split
p = 0.7
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = p, random_state = 42)

# 2. Naive Bayes from scratch

##2.1. Likelyhood function

Here we will difine a __likelihood function__, that will returns the __probability__ of a given input __y__ to belong to a particular distribuion of examples givem by __Z__. <br/>
Initially, let's define a function to compute the __joint probability density function__ (PDF):

$P(x|C_i)= \prod _{j=1} ^{d} P(x_j|C_i),  \space \space \space i=1,2,...,k.$

where $C_i$ are the classes. <br/>
Now, considering the __Gaussian (or Normal) distribution__, each feature $x_j$ has the following PDF for each class:

$P(x_j|C_i)=\frac{1}{\sqrt{2\pi\sigma C_i}}e^{-\frac{1}{2}.\left( \frac{x_j- \mu _{C_i}}{\sigma C_i} \right)^2 }, \space \space \space i=1,2,...,k.$



In [4]:
def likelyhood(y, Z):
    def gaussian(x, mu, sig):
        return np.exp(-np.power(x - mu, 2.) / (2 * np.power(sig, 2.)))
    prob = 1
    for j in np.arange(0, Z.shape[1]):
        m = np.mean(Z[:,j])
        s = np.std(Z[:,j])
        prob = prob*gaussian(y[j], m, s)
    return prob



##2.2. Class estimation

Now, let's estimate the probability of our __trainning data__ to belong to __each of the classes__...

In [5]:
P = pd.DataFrame(data=np.zeros((X_test.shape[0], len(classes))), columns = classes)
for i in np.arange(0, len(classes)):
    elements = tuple(np.where(y_train == classes[i]))
    Z = X_train[elements,:][0]
    for j in np.arange(0,X_test.shape[0]):
        x = X_test[j,:]
        pj = likelyhood(x,Z)
        P[classes[i]][j] = pj*len(elements)/X_train.shape[0]

/tmp/ipykernel_1817/831102427.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  P[classes[i]][j] = pj*len(elements)/X_train.shape[0]


Here we can see the probability of each one of the __training set__ elements to belong to each class...

In [6]:
P.head(10)

,Setosa,Versicolor,Virginica
0,3.477446e-91,4.440479e-03,4.107993e-05
1,1.712308e-04,1.196823e-16,4.171552e-23
2,2.115370e-290,3.100666e-17,2.363765e-05
3,7.422472e-95,4.042876e-03,2.146958e-04
4,1.973245e-107,8.057427e-04,3.352704e-04
5,1.475904e-03,5.335375e-15,1.159063e-22
6,1.695635e-54,3.230445e-03,2.400042e-07
7,1.048899e-175,6.868359e-10,3.319537e-03
8,1.406287e-97,8.399067e-04,1.117962e-05
9,1.290511e-60,6.780257e-03,6.579053e-07


## 2.3. Accuracy

Let's check the __accuracy__ of our system using the __test set__...

In [7]:
from sklearn.metrics import accuracy_score

y_pred = []
for i in np.arange(0, P.shape[0]):
    c = np.argmax(np.array(P.iloc[[i]]))
    y_pred.append(P.columns[c])
y_pred = np.array(y_pred, dtype=str)


score = accuracy_score(y_pred, y_test)
print('Accuracy:', score)

Accuracy: 0.9555555555555556



# 3. Naive Bayes from Scikit-Learn


##3.1. Gaussian distribution

We can use the scikit-learn library to easily implement our classifier....

In [8]:
# Naive Bayes from scikit-lear

from sklearn.naive_bayes import GaussianNB
from sklearn import metrics

model = GaussianNB() # <--- selected distribution curve: GaussianNB(), BernoulliNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = accuracy_score(y_pred, y_test)
print('Accuracy:', score)

Accuracy: 0.9777777777777777


#3.2. Bernoulli distribution

We can try other distributions to model our data...

In [9]:
from sklearn.naive_bayes import BernoulliNB

model = BernoulliNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = accuracy_score(y_pred, y_test)
print('Accuracy:', score)

Accuracy: 0.28888888888888886


____

<center>
<img src="https://upload.wikimedia.org/wikipedia/commons/0/0a/Logo_Unesp.svg" width="400" style="float: left; margin-right: 5px;" border="0px" />
</center>